# ORT: Gravity Advanced

**Chapter 12, part 2 | §12.15–12.21, §12.26–12.27 | Formulas 40–65, 82–103**

This notebook covers the advanced gravitational effects:
Reissner-Nordström (charged mass), Shapiro delay, geodetic precession,
photon sphere, Einstein rings, the S2 star, gravitational waves and the Kerr metric.

---

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parent / 'shared'))
from spacetime_core import *
from spacetime_plots import (cosmological_shell_diagram, gw_strain_plot, gw_inspiral_interactive,
    kerr_geometry_plot, kerr_frame_drag_field, isco_comparison_plot, comparison_table,
    photon_sphere_shadow, einstein_ring_plot)
import matplotlib.pyplot as plt
import math
import numpy as np
%matplotlib inline

## §12.15 — Reissner-Nordström (Charged Mass)

The metric function with charge term:

$$f(r) = 1 - \frac{r_s}{r} + \frac{r_Q^2}{r^2} \qquad (40)$$

with $r_Q^2 = k_e Q^2 G / c^4$.

**Two horizons**:

$$r_{\pm} = \frac{r_s \pm \sqrt{r_s^2 - 4r_Q^2}}{2} \qquad (42)$$

**Extremal charge** ($r_+ = r_-$):

$$Q_{ext} = c^2 \sqrt{\frac{r_s^2}{4 k_e G}} \qquad (43)$$

Charge **counteracts** gravity: light deflection, precession, $v_{grav}$ all decrease.

In [ ]:
# Reissner-Nordström: comparison Schwarzschild vs charged
M_BH = 10 * M_SUN
bh_sch = GravityModel(M_BH)               # Uncharged
Q_ext = bh_sch.extremal_charge()
bh_half = GravityModel(M_BH, charge=0.5 * Q_ext)  # Half extremal charge
bh_ext = GravityModel(M_BH, charge=0.99 * Q_ext)  # Near extremal

print(f"=== Reissner-Nordström (10 M☉) ===")
print(f"r_s = {bh_sch.rs:.3e} m")
print(f"Q_ext = {Q_ext:.3e} C")
print()

for label, model in [("Q = 0 (Schwarzschild)", bh_sch),
                      ("Q = 0.5 Q_ext", bh_half),
                      ("Q = 0.99 Q_ext", bh_ext)]:
    r_plus = model.r_plus
    r_minus = model.r_minus
    print(f"--- {label} ---")
    print(f"  r+ = {r_plus:.3e} m = {r_plus/bh_sch.rs:.4f} r_s")
    print(f"  r- = {r_minus:.3e} m = {r_minus/bh_sch.rs:.4f} r_s")
    # Light deflection at b = 10 r_s
    b = 10 * bh_sch.rs
    alpha = model.light_deflection_arcsec(b)
    print(f"  Light deflection (b=10r_s): {alpha:.6f}\"")
    print()

## §12.16 — Shapiro Delay

The fourth classical test of GRT. A signal passing near a mass is delayed:

$$\Delta t = \frac{r_s}{c} \cdot \ln\!\left(\frac{4 r_1 r_2}{b^2}\right) \qquad (49)$$

With charge correction (RN):

$$\Delta t_{RN} = \frac{r_s}{c} \cdot \ln\!\left(\frac{4 r_1 r_2}{b^2}\right) - \frac{\pi r_Q^2}{c \cdot b} \qquad (50)$$

As with light deflection: **50/50 temporal + spatial**.

In [ ]:
# Shapiro delay: Cassini measurement
b_cassini = 1.6 * R_SUN       # impact parameter
r_earth = A_EARTH_ORBIT        # Earth-Sun distance
r_saturn = R_SATURN_ORBIT      # Saturn-Sun distance

delay = SUN.shapiro_delay(r_earth, r_saturn, b_cassini)
delay_us = delay * 1e6
delay_roundtrip = SUN.shapiro_delay_roundtrip(r_earth, r_saturn, b_cassini)

# Temporal only (half)
delay_half = SUN.half_shapiro_delay(r_earth, r_saturn, b_cassini)

print("=== Shapiro Delay (Cassini) ===")
print(f"b = {b_cassini/R_SUN:.1f} R_sun = {b_cassini:.3e} m")
print(f"r₁ (Earth)  = {r_earth:.3e} m")
print(f"r₂ (Saturn) = {r_saturn:.3e} m")
print()
print(f"Temporal only (half):     {delay_half*1e6:.2f} µs")
print(f"Full (temp+spatial):      {delay_us:.2f} µs")
print(f"Round trip:               {delay_roundtrip*1e6:.2f} µs")
print()
print(f"Cassini result (2003): γ = 1 + (2.1 ± 2.3) × 10⁻⁵")

## §12.17 — Geodetic (de Sitter) Precession

The spin axis of a gyroscope precesses in orbit around a mass:

$$\Delta\theta = 2\pi \left[1 - \sqrt{1 - \frac{3r_s}{2r}}\right] \qquad (51)$$

Weak-field approximation:

$$\Delta\theta \approx \frac{3\pi}{2} \cdot \frac{r_s}{r} = \frac{3\pi GM}{rc^2} \qquad (52)$$

With charge correction (RN):

$$\Delta\theta = 2\pi \left[1 - \sqrt{1 - \frac{3r_s}{2r} + \frac{2r_Q^2}{r^2}}\right] \qquad (53)$$

### The 1/3 + 2/3 Split

Unlike the other effects (50/50):
- **1/3 — Thomas precession** (SRT, temporal)
- **2/3 — Spatial curvature** (GR, spatial)

| Effect | Temporal | Spatial | Ratio |
|--------|----------|---------|-------|
| Light deflection | 50% | 50% | 1:1 |
| Shapiro delay | 50% | 50% | 1:1 |
| Orbital precession | 50% | 50% | 1:1 |
| Geodetic precession | 33% | 67% | 1:2 |

In [ ]:
# Geodetic precession: Gravity Probe B
gpb_period = EARTH.orbital_period(GPB_ORBIT_RADIUS)
gpb_prec = EARTH.geodetic_precession(GPB_ORBIT_RADIUS)
gpb_mas_yr = EARTH.geodetic_precession_mas_per_year(GPB_ORBIT_RADIUS, gpb_period)

print("=== Geodetic Precession — Gravity Probe B ===")
print(f"Orbital altitude: 642 km (r = {GPB_ORBIT_RADIUS:.3e} m)")
print(f"Orbital period: {gpb_period:.1f} s = {gpb_period/60:.1f} min")
print(f"\nΔθ per orbit        = {gpb_prec:.6e} rad")
print(f"Δθ per year         = {gpb_mas_yr:.1f} mas/yr")
print(f"\nPredicted (GRT):   6606.1 mas/yr")
print(f"Measured (GP-B):   6601.8 ± 18.3 mas/yr")
print()

# 1/3 + 2/3 split
thomas = gpb_mas_yr / 3
curvature = 2 * gpb_mas_yr / 3
print(f"Thomas contribution (1/3):    {thomas:.1f} mas/yr")
print(f"Curvature contribution (2/3): {curvature:.1f} mas/yr")

## §12.18 — Model Comparison

Six models, ten effects — the **honesty table**.

Legend: **✓** correct, **½** half value, **✗** no prediction, **—** n/a, **⚠** limited

In [ ]:
# Model comparison table
comparison_table(lang='en')

## §12.19 — Photon Sphere and Black Hole Shadow (EHT)

The photon sphere is the radius where photons orbit in an (unstable) circle:

$$r_{photon} = \frac{3}{2} r_s = \frac{3GM}{c^2} \qquad (55)$$

For RN:

$$r_{photon} = \frac{3r_s + \sqrt{9r_s^2 - 32r_Q^2}}{4} \qquad (56)$$

The **shadow radius** (critical impact parameter):

$$b_{crit} = \frac{r_{photon}}{\sqrt{f(r_{photon})}} \qquad (57)$$

For Schwarzschild: $b_{crit} = \frac{3\sqrt{3}}{2} r_s \approx 2.598\, r_s$

The angular diameter of the shadow:

$$\theta_{shadow} = \frac{2 b_{crit}}{D} \qquad (58)$$

In [ ]:
# Photon sphere and shadow visualization
fig = photon_sphere_shadow(lang='en')
plt.show()

In [ ]:
# Shadow size M87* and Sgr A*
print("=== Black Hole Shadows (EHT) ===")
print()

for name, model, dist in [("M87*", M87_STAR, D_M87_STAR),
                           ("Sgr A*", SGR_A_STAR, D_SGR_A_STAR)]:
    r_ph = model.photon_sphere()
    b_crit = model.shadow_radius()
    theta_uas = model.shadow_angular_diameter(dist)
    print(f"--- {name} ---")
    print(f"  Mass:         {model.mass/M_SUN:.2e} M☉")
    print(f"  Distance:     {dist/PARSEC:.0f} pc")
    print(f"  r_s:          {model.rs:.3e} m")
    print(f"  r_photon:     {r_ph:.3e} m = {r_ph/model.rs:.3f} r_s")
    print(f"  b_crit:       {b_crit:.3e} m = {b_crit/model.rs:.3f} r_s")
    print(f"  Shadow:       {theta_uas:.1f} µas")
    print()

print("EHT measurements:")
print("  M87*   (2019): 42 ± 3 µas")
print("  Sgr A* (2022): 51.8 ± 2.3 µas")

## §12.20 — Einstein Rings

With perfect alignment, the light forms a ring with angular radius:

$$\theta_E = \sqrt{\frac{2r_s \cdot D_{LS}}{D_L \cdot D_S}} = \sqrt{\frac{4GM}{c^2} \cdot \frac{D_{LS}}{D_L \cdot D_S}} \qquad (60)$$

Point-mass magnification:

$$\mu = \frac{u^2 + 2}{u \sqrt{u^2 + 4}} \qquad (61)$$

with $u = \beta / \theta_E$ (normalized source position).

In [ ]:
# Einstein ring example
d_L = 1.0e9 * PARSEC    # 1 Gpc (lens)
d_S = 2.0e9 * PARSEC    # 2 Gpc (source)
d_LS = 1.0e9 * PARSEC   # distance lens-source
lens = GravityModel(1e12 * M_SUN)  # 10^12 M_sun cluster

theta_E = lens.einstein_ring_angle_arcsec(d_L, d_S, d_LS)

print("=== Einstein Ring ===")
print(f"Lens mass:   10¹² M☉ (galaxy cluster)")
print(f"D_L = 1 Gpc, D_S = 2 Gpc, D_LS = 1 Gpc")
print(f"\nθ_E = {theta_E:.2f} arcseconds")
print()

# Magnification at various source positions
print("Magnification at offset from axis:")
for u in [0.1, 0.5, 1.0, 2.0, 5.0]:
    mu = GravityModel.lens_magnification(u)
    print(f"  u = {u:.1f}:  µ = {mu:.3f}")

In [ ]:
# Einstein ring plot
fig = einstein_ring_plot(theta_E_arcsec=theta_E, lang='en')
plt.show()

## §12.21 — Strong-Field Redshift: the S2 Star at Sgr A*

The S2 star orbits Sgr A* ($4 \times 10^6 M_\odot$) in ~16 years. At pericenter:

**Gravitational redshift:**
$$z_{grav} = \frac{1}{\sqrt{1 - r_s/r_p}} - 1 \approx \frac{r_s}{2r_p} \qquad (62)$$

**Transverse Doppler shift (SRT):**
$$z_{SRT} = \frac{1}{\sqrt{1 - v^2/c^2}} - 1 \approx \frac{v^2}{2c^2} \qquad (63)$$

**Combined (ORT):**
$$z_{total} = \frac{1}{\sqrt{f(r) - v^2/c^2}} - 1 \qquad (64)$$

**Schwarzschild precession of S2:**
$$\Delta\varphi = \frac{3\pi r_s}{a(1-e^2)} \approx 0.19\degree \approx 12' \text{ per orbit} \qquad (65)$$

In [ ]:
# S2 star at Sgr A*
r_peri = GravityModel.pericenter_distance(A_S2, E_S2)
v_peri = SGR_A_STAR.pericenter_velocity(A_S2, E_S2)

print("=== S2 Star at Sgr A* ===")
print(f"Semi-major axis a = {A_S2:.3e} m = {A_S2/1.496e11:.0f} AU")
print(f"Eccentricity e    = {E_S2}")
print(f"Orbital period    = {P_S2/(365.25*86400):.2f} years")
print(f"Pericenter dist.  = {r_peri:.3e} m = {r_peri/1.496e11:.0f} AU")
print(f"r_p / r_s         = {r_peri/SGR_A_STAR.rs:.0f}")
print(f"Pericenter vel.   = {v_peri:.0f} m/s = {v_peri/1000:.0f} km/s = {v_peri/C:.4f} c")
print()

# Redshifts
z_grav = 1/SGR_A_STAR.time_dilation_factor(r_peri) - 1
z_srt = 1/math.sqrt(1 - (v_peri/C)**2) - 1
z_combined = SGR_A_STAR.combined_redshift(r_peri, v_peri)

print("--- Redshifts at pericenter ---")
print(f"z_grav (gravitational)   = {z_grav:.4e}  (Δv = {z_grav*C/1000:.0f} km/s)")
print(f"z_SRT  (transverse)      = {z_srt:.4e}  (Δv = {z_srt*C/1000:.0f} km/s)")
print(f"z_total (combined)       = {z_combined:.4e}  (Δv = {z_combined*C/1000:.0f} km/s)")
print()

# Schwarzschild precession
prec_s2 = SGR_A_STAR.orbital_precession(A_S2, E_S2)
prec_s2_arcmin = prec_s2 * (180/math.pi) * 60

print("--- Schwarzschild precession ---")
print(f"Δφ per orbit = {prec_s2:.6e} rad = {prec_s2_arcmin:.2f}' = {prec_s2*180/math.pi:.3f}°")
print()
print("GRAVITY/ESO measurements:")
print("  Redshift (2018):    f = 0.88 ± 0.17 (GRT: f = 1)")
print("  Precession (2020):  f_SP = 1.10 ± 0.19 (GRT: f_SP = 1)")

---

## Gravitational Waves and Kerr Metric

The following sections come from §12.26–12.27 of MODEL.md.

---

## Summary

This notebook covers the advanced gravitational effects of ORT:

| §  | Topic | Formulas | Status |
|---|---|---|---|
| 12.15 | Reissner-Nordström | (40)-(43) | Derived |
| 12.16 | Shapiro delay | (49)-(50) | Confirmed (Cassini) |
| 12.17 | Geodetic precession | (51)-(53) | Confirmed (GP-B) |
| 12.18 | Model comparison | — | Honesty table |
| 12.19 | Photon sphere & EHT | (55)-(58) | Confirmed (M87*, Sgr A*) |
| 12.20 | Einstein rings | (60)-(61) | Observed |
| 12.21 | S2 star redshift | (62)-(65) | Confirmed (GRAVITY) |
| 12.26 | Gravitational waves | (82)-(96) | Confirmed (LIGO) |
| 12.27 | Kerr metric | (97)-(103) | Confirmed (GP-B, LAGEOS) |

**Next**: Notebook 04 covers cosmology (§12.22-12.25).